# Checked answer recovery: when is the whole curve defined?

This is an additive practice exercise for [12.3 Mini VLM](../../../chapter12_vlm_interpretability/exercises/part3_mini_vlm_from_scratch/12.3_Mini_VLM_from_Scratch_exercises.ipynb).
The existing saved historical Mini reference uses its original unguarded normalization. Its source, figures, recipes and results are unchanged. This addon makes a different, explicit prospective failure policy visible for your own future runs. No new model or topic is introduced.

The numerical quantity is still `(LD_patched - LD_corrupt) / (LD_clean - LD_corrupt)`. Your own completed `logit_diff_recovery` supplies it. You will add checks around that callback, not replace it with an imported solution.

**Forecast before looking.** Which pairs would make this recovery undefined or reverse its intended contrast? What can a very small positive denominator do? Record a prediction, confidence (0–100%), reasoning, and whether you have already seen the saved curve. Do this before opening the solution dropdowns.

Use the existing [12.3 guided prediction workbook](../../units/extension_predictions/exercises.ipynb#section-12-3) alongside this exercise. Seeing a saved answer is preparation, not independent mastery.

**Scope.** The fixed tiny CPU exercises below need no model, data, RNG or training. The final learner-run cells require a separately authorized, already prepared 12.3 session. Nothing here automatically loads, reconstructs or trains a missing model. Saved reference execution does not establish fresh reproduction or human learner acceptance.

In [ ]:
# Tiny CPU exercise setup only. Do not run the historical lesson's setup.
import torch as t

## 1. Define the failure contract

Stop the whole selected cohort for a nonfinite baseline logit difference or gap, a zero gap (including negative zero), a reversed gap, or a nonfinite recovery. Report every failing original pair index in the checked condition before any mean or figure. Do not drop, clamp, floor, take absolute values, reweight or retry pairs.

Every finite strictly positive gap stays, however small. Every finite recovery stays, including negative values and values above one. Keep each baseline tensor's dtype and the metric callback's own floating dtype. Display the full gap vector for interpretation rather than threshold-selecting “small” rows.

The visible bookkeeping below gives errors their pair IDs. It is ordinary structure validation, not a hidden metric. Pair IDs must be a nonempty unique tuple of original nonnegative integer indices. Malformed identity metadata raises a structural error without inventing replacement IDs. IDs used for tensor indexing must be int32 or int64 on the logits' device.

In [ ]:
class RecoveryCohortFailure(ValueError):
    """No curves are returned when any part of the selected cohort fails."""
    def __init__(self, *, stage, pair_indices, invalid_pairs, condition=None):
        self.stage = stage
        self.cohort_pair_indices = tuple(pair_indices)
        self.invalid_pairs = tuple(invalid_pairs)
        self.reasons = tuple(row["reason"] for row in invalid_pairs)
        bad = {row["pair_index"] for row in invalid_pairs}
        self.pair_indices = tuple(index for index in pair_indices if index in bad)
        self.condition = condition
        self.attempted_conditions = ()
        self.completed_conditions = ()
        self.unattempted_conditions = ()
        super().__init__(
            f"Complete cohort stopped: {stage}; pairs={self.pair_indices}; "
            f"condition={condition}; failures={self.invalid_pairs}"
        )


def _stop_recovery(stage, pair_indices, reason, condition=None):
    # Invalid identity metadata cannot be converted into invented pair IDs.
    indices = tuple(pair_indices) if isinstance(pair_indices, (tuple, list)) else ()
    valid_ids = tuple(i for i in indices if type(i) is int and i >= 0)
    raise RecoveryCohortFailure(
        stage=stage, pair_indices=valid_ids, condition=condition,
        invalid_pairs=[{"pair_index": i, "reason": reason} for i in valid_ids],
    )


def _validate_recovery_structure(clean_logits, corrupt_logits, clean_ids,
                                 corrupt_ids, pair_indices):
    if (not isinstance(pair_indices, tuple) or not pair_indices
            or any(type(i) is not int or i < 0 for i in pair_indices)
            or len(set(pair_indices)) != len(pair_indices)):
        _stop_recovery("structure", pair_indices, "invalid_pair_indices")
    if not isinstance(clean_logits, t.Tensor) or not isinstance(corrupt_logits, t.Tensor):
        _stop_recovery("structure", pair_indices, "invalid_shape_or_type")
    if (clean_logits.ndim != 2 or clean_logits.shape[0] != len(pair_indices)
            or clean_logits.shape[1] == 0 or not clean_logits.is_floating_point()
            or corrupt_logits.shape != clean_logits.shape
            or corrupt_logits.dtype != clean_logits.dtype
            or corrupt_logits.device != clean_logits.device):
        _stop_recovery("structure", pair_indices, "invalid_shape_or_type")
    for ids in (clean_ids, corrupt_ids):
        if (not isinstance(ids, t.Tensor) or ids.shape != (len(pair_indices),)
                or ids.dtype not in (t.int32, t.int64) or ids.device != clean_logits.device):
            _stop_recovery("structure", pair_indices, "invalid_shape_or_type")
        if ((ids < 0) | (ids >= clean_logits.shape[1])).any().item():
            _stop_recovery("structure", pair_indices, "invalid_token_ids")

## 2. Exercise — validate the whole baseline cohort

**Difficulty:** 2/5 · **Importance:** 5/5 · **Time:** 10 minutes.

Implement the function below. Extract each clean-answer minus corrupt-answer logit difference in both runs, then their gap. Collect all failures in original pair order and raise one `RecoveryCohortFailure(stage="baseline", ...)`. Return detached clones of the three complete vectors on success. Check the components as well as subtraction overflow; no scientific dtype conversion is allowed.

<details><summary>Hint and expected output</summary>

With clean logits `[[3,0],[2,0]]`, corrupt logits `[[-1,0],[0,0]]`, and answer IDs `[0,0]` versus `[1,1]`, the ordered table is `pair 7: gap 4`, `pair 9: gap 2`. After setting clean row 9 to `[0,0]`, the whole cohort stops and reports pair 9. A finite tiny positive gap does not fail.

Use `rows = t.arange(clean_logits.shape[0], device=clean_logits.device)`, then the two indexed logit differences. Check `t.isfinite` on both components and their difference, plus equality to zero and a negative finite gap. A single bad row invalidates the whole result.
</details>

In [ ]:
def validate_recovery_baselines(clean_logits, corrupt_logits, clean_ids,
                                corrupt_ids, *, pair_indices):
    _validate_recovery_structure(
        clean_logits, corrupt_logits, clean_ids, corrupt_ids, pair_indices
    )
    raise NotImplementedError("Implement whole-cohort baseline validation inline")

In [ ]:
# Independent literal CPU fixtures only: immediate tests for YOUR validator.
def test_my_baseline_guard(validate):
    clean = t.tensor([[3., 0.], [2., 0.]], dtype=t.float64, device="cpu")
    corrupt = t.tensor([[-1., 0.], [0., 0.]], dtype=t.float64, device="cpu")
    a, b = t.tensor([0, 0]), t.tensor([1, 1])
    assert validate(clean, corrupt, a, b, pair_indices=(7, 9))[2].tolist() == [4., 2.]
    for bad in (0., -0., -1., float("nan"), float("inf"), -float("inf")):
        clean[1, 0] = bad
        try:
            validate(clean, corrupt, a, b, pair_indices=(7, 9))
        except RecoveryCohortFailure as error:
            assert error.stage == "baseline" and error.pair_indices == (9,)
        else:
            raise AssertionError("An invalid pair escaped the baseline check")
    clean[1, 0] = 1e-300
    gap = validate(clean, corrupt, a, b, pair_indices=(7, 9))[2]
    assert gap.shape == (2,) and gap[1].item() > 0
    print("Baseline controls passed; the tiny positive row was kept.")


test_my_baseline_guard(validate_recovery_baselines)

<details data-guard-solution="baseline-solution">
<summary>Solution — inspect after trying</summary>

```python
class RecoveryCohortFailure(ValueError):
    """No curves are returned when any part of the selected cohort fails."""
    def __init__(self, *, stage, pair_indices, invalid_pairs, condition=None):
        self.stage = stage
        self.cohort_pair_indices = tuple(pair_indices)
        self.invalid_pairs = tuple(invalid_pairs)
        self.reasons = tuple(row["reason"] for row in invalid_pairs)
        bad = {row["pair_index"] for row in invalid_pairs}
        self.pair_indices = tuple(index for index in pair_indices if index in bad)
        self.condition = condition
        self.attempted_conditions = ()
        self.completed_conditions = ()
        self.unattempted_conditions = ()
        super().__init__(
            f"Complete cohort stopped: {stage}; pairs={self.pair_indices}; "
            f"condition={condition}; failures={self.invalid_pairs}"
        )


def _stop_recovery(stage, pair_indices, reason, condition=None):
    # Invalid identity metadata cannot be converted into invented pair IDs.
    indices = tuple(pair_indices) if isinstance(pair_indices, (tuple, list)) else ()
    valid_ids = tuple(i for i in indices if type(i) is int and i >= 0)
    raise RecoveryCohortFailure(
        stage=stage, pair_indices=valid_ids, condition=condition,
        invalid_pairs=[{"pair_index": i, "reason": reason} for i in valid_ids],
    )


def _validate_recovery_structure(clean_logits, corrupt_logits, clean_ids,
                                 corrupt_ids, pair_indices):
    if (not isinstance(pair_indices, tuple) or not pair_indices
            or any(type(i) is not int or i < 0 for i in pair_indices)
            or len(set(pair_indices)) != len(pair_indices)):
        _stop_recovery("structure", pair_indices, "invalid_pair_indices")
    if not isinstance(clean_logits, t.Tensor) or not isinstance(corrupt_logits, t.Tensor):
        _stop_recovery("structure", pair_indices, "invalid_shape_or_type")
    if (clean_logits.ndim != 2 or clean_logits.shape[0] != len(pair_indices)
            or clean_logits.shape[1] == 0 or not clean_logits.is_floating_point()
            or corrupt_logits.shape != clean_logits.shape
            or corrupt_logits.dtype != clean_logits.dtype
            or corrupt_logits.device != clean_logits.device):
        _stop_recovery("structure", pair_indices, "invalid_shape_or_type")
    for ids in (clean_ids, corrupt_ids):
        if (not isinstance(ids, t.Tensor) or ids.shape != (len(pair_indices),)
                or ids.dtype not in (t.int32, t.int64) or ids.device != clean_logits.device):
            _stop_recovery("structure", pair_indices, "invalid_shape_or_type")
        if ((ids < 0) | (ids >= clean_logits.shape[1])).any().item():
            _stop_recovery("structure", pair_indices, "invalid_token_ids")


def validate_recovery_baselines(clean_logits, corrupt_logits, clean_ids,
                                corrupt_ids, *, pair_indices):
    """Keep every finite positive gap; raise once with all invalid original IDs."""
    _validate_recovery_structure(
        clean_logits, corrupt_logits, clean_ids, corrupt_ids, pair_indices
    )
    rows = t.arange(clean_logits.shape[0], device=clean_logits.device)
    clean_ld = clean_logits[rows, clean_ids] - clean_logits[rows, corrupt_ids]
    corrupt_ld = corrupt_logits[rows, clean_ids] - corrupt_logits[rows, corrupt_ids]
    gap = clean_ld - corrupt_ld
    checks = (
        ("nonfinite_clean_ld", ~t.isfinite(clean_ld)),
        ("nonfinite_corrupt_ld", ~t.isfinite(corrupt_ld)),
        ("nonfinite_gap", ~t.isfinite(gap)),
        ("zero_gap", gap == 0),
        ("reversed_gap", t.isfinite(gap) & (gap < 0)),
    )
    failures = [
        {"pair_index": pair_index, "reason": reason}
        for row, pair_index in enumerate(pair_indices)
        for reason, mask in checks if mask[row].item()
    ]
    if failures:
        raise RecoveryCohortFailure(
            stage="baseline", pair_indices=pair_indices, invalid_pairs=failures
        )
    return clean_ld.detach().clone(), corrupt_ld.detach().clone(), gap.detach().clone()

```

</details>

## 3. Exercise — check your own metric callback

**Difficulty:** 2/5 · **Importance:** 5/5 · **Time:** 10 minutes.

Validate baselines before invoking `metric_callback(patched_logits, clean_logits, corrupt_logits, clean_ids, corrupt_ids)` exactly once. Do not compute a replacement metric. Require the returned tensor to have shape `(N,)` and a floating dtype and the same device as the logits; preserve the callback's own dtype without conversion. Reject every nonfinite recovery with original pair indices and the condition. Clone the detached result immediately so a later callback cannot overwrite a reused buffer.

The callback must implement the original formula correctly and must not mutate its input tensors or other experiment state. This guard is not a sandbox for arbitrary callback behavior. The existing clean/corrupt/interpolation/hand-worked tests remain necessary; the guard does not certify metric correctness.

<details><summary>Hint</summary>

Collect offending rows with `not t.isfinite(recovery[row]).item()`. There is no allowed-interval check: `-0.25` and `1.25` are finite, valid recoveries. Preserve a callback exception as the cause of a whole-cohort failure; do not fall back to another implementation.
</details>

In [ ]:
def checked_learner_recovery(patched_logits, clean_logits, corrupt_logits,
                             clean_ids, corrupt_ids, *, pair_indices,
                             metric_callback, condition):
    raise NotImplementedError("Validate, call your metric once, check all rows, clone")

In [ ]:
def test_my_metric_guard(check):
    clean = t.tensor([[3., 0.], [2., 0.]], dtype=t.float32, device="cpu")
    corrupt = t.tensor([[-1., 0.], [0., 0.]], dtype=t.float32, device="cpu")
    a, b = t.tensor([0, 0]), t.tensor([1, 1])
    buffer = t.tensor([-0.25, 1.25], dtype=t.float32, device="cpu")
    calls = []
    def dev_metric(*args):
        calls.append(1)
        return buffer
    out = check(clean, clean, corrupt, a, b, pair_indices=(7, 9),
                metric_callback=dev_metric, condition=(0, "object"))
    assert calls == [1] and out.tolist() == [-0.25, 1.25]
    buffer.fill_(99.)
    assert out.tolist() == [-0.25, 1.25]
    buffer[1] = float("inf")
    try:
        check(clean, clean, corrupt, a, b, pair_indices=(7, 9),
              metric_callback=dev_metric, condition=(0, "object"))
    except RecoveryCohortFailure as error:
        assert error.stage == "metric" and error.pair_indices == (9,)
    else:
        raise AssertionError("Nonfinite callback result escaped")
    print("Own callback, outside-range values, buffer clone and failure checks passed.")


test_my_metric_guard(checked_learner_recovery)

<details data-guard-solution="metric-solution">
<summary>Solution — inspect after trying</summary>

```python
def checked_learner_recovery(patched_logits, clean_logits, corrupt_logits,
                             clean_ids, corrupt_ids, *, pair_indices,
                             metric_callback, condition):
    """Call the learner's metric once, keep its dtype and rows, check before mean."""
    validate_recovery_baselines(
        clean_logits, corrupt_logits, clean_ids, corrupt_ids, pair_indices=pair_indices
    )
    if (not isinstance(patched_logits, t.Tensor)
            or patched_logits.shape != clean_logits.shape
            or patched_logits.dtype != clean_logits.dtype
            or patched_logits.device != clean_logits.device or not callable(metric_callback)):
        _stop_recovery("structure", pair_indices, "invalid_shape_or_type", condition)
    try:
        recovery = metric_callback(
            patched_logits, clean_logits, corrupt_logits, clean_ids, corrupt_ids
        )
    except Exception as error:
        try:
            _stop_recovery("callback_exception", pair_indices, "callback_exception", condition)
        except RecoveryCohortFailure as failure:
            raise failure from error
    if (not isinstance(recovery, t.Tensor) or recovery.shape != (len(pair_indices),)
            or not recovery.is_floating_point() or recovery.device != clean_logits.device):
        _stop_recovery("structure", pair_indices, "invalid_shape_or_type", condition)
    failures = [
        {"pair_index": pair_index, "reason": "nonfinite_recovery"}
        for row, pair_index in enumerate(pair_indices) if not t.isfinite(recovery[row]).item()
    ]
    if failures:
        raise RecoveryCohortFailure(
            stage="metric", pair_indices=pair_indices, invalid_pairs=failures, condition=condition
        )
    # A callback may reuse an output buffer on its next call. Preserve this row now.
    return recovery.detach().clone()

```

</details>

## 4. Exercise — collect everything, then aggregate

**Difficulty:** 3/5 · **Importance:** 5/5 · **Time:** 15 minutes.

Write a separate `checked_learner_sweep`. Keep layer order and mask insertion order. Validate baselines before the first patch or metric callback. Collect all checked per-pair vectors before computing **any** mean. Then compute means without changing dtype, check each is finite, and return the curves only when every mean passes. Do not plot or write files inside this function.

If any condition fails, invalidate the whole sweep. Report original bad pair IDs, the condition, completed/attempted conditions, and later unattempted conditions. Earlier valid vectors do not become an accepted partial curve. A mean can overflow even when all inputs are finite: in that case identify all contributing pair IDs and publish no curves. Do not silently substitute a different reduction.

<details><summary>Hint and expected result</summary>

Use two loops: one to collect checked vectors, one to form means. For two layers and masks ordered object/random, callbacks returning `[1,1]`, `[2,2]`, `[3,3]`, `[4,4]` give `object: [1,3]`, `random: [2,4]`. A failure in the third callback must occur before every mean, leaving the fourth condition unattempted.
</details>

In [ ]:
def checked_learner_sweep(*, layers, masks, pair_indices, clean_logits,
                          corrupt_logits, clean_ids, corrupt_ids,
                          patch_logits_callback, metric_callback):
    raise NotImplementedError("Collect every checked row before any mean")

In [ ]:
def test_my_sweep_guard(sweep):
    clean = t.tensor([[3., 0.], [2., 0.]], dtype=t.float32, device="cpu")
    corrupt = t.tensor([[-1., 0.], [0., 0.]], dtype=t.float32, device="cpu")
    a, b = t.tensor([0, 0]), t.tensor([1, 1])
    calls = []
    def dev_patch(layer, name):
        calls.append((layer, name))
        return clean
    def dev_metric(*args):
        return t.tensor([0., float("inf") if len(calls) == 3 else 1.])
    # A temporary recorder proves no mean runs on a late row failure.
    from unittest.mock import patch
    def forbid_mean(*args, **kwargs):
        raise AssertionError("A mean ran before the complete sweep passed")
    with patch.object(t.Tensor, "mean", forbid_mean):
        try:
            sweep(layers=(0, 1), masks={"object": None, "random": None},
                  pair_indices=(7, 9), clean_logits=clean, corrupt_logits=corrupt,
                  clean_ids=a, corrupt_ids=b, patch_logits_callback=dev_patch,
                  metric_callback=dev_metric)
        except RecoveryCohortFailure as error:
            assert error.condition == (1, "object") and error.pair_indices == (9,)
            assert error.completed_conditions == ((0, "object"), (0, "random"))
            assert error.unattempted_conditions == ((1, "random"),)
        else:
            raise AssertionError("The complete cohort should have stopped")
    assert len(calls) == 3
    print("Late failure stopped the whole sweep before every mean and figure.")


test_my_sweep_guard(checked_learner_sweep)

<details data-guard-solution="sweep-solution">
<summary>Solution — inspect after trying</summary>

```python
def checked_learner_sweep(*, layers, masks, pair_indices, clean_logits,
                          corrupt_logits, clean_ids, corrupt_ids,
                          patch_logits_callback, metric_callback):
    """Validate the full condition grid before any mean; return all curves or fail."""
    if (not isinstance(layers, (tuple, list)) or not layers
            or any(type(layer) is not int or layer < 0 for layer in layers)
            or len(set(layers)) != len(layers)
            or not isinstance(masks, dict) or not masks
            or any(not isinstance(name, str) or not name for name in masks)
            or not callable(patch_logits_callback) or not callable(metric_callback)):
        _stop_recovery("structure", pair_indices, "invalid_condition_grid_or_callback")
    condition_order = tuple((layer, name) for layer in layers for name in masks)
    attempted, completed, records = [], [], []
    condition = None
    try:
        clean_ld, corrupt_ld, gap = validate_recovery_baselines(
            clean_logits, corrupt_logits, clean_ids, corrupt_ids, pair_indices=pair_indices
        )
        for condition in condition_order:
            attempted.append(condition)
            try:
                patched = patch_logits_callback(*condition)
            except Exception as error:
                try:
                    _stop_recovery("callback_exception", pair_indices, "callback_exception", condition)
                except RecoveryCohortFailure as failure:
                    raise failure from error
            recovery = checked_learner_recovery(
                patched, clean_logits, corrupt_logits, clean_ids, corrupt_ids,
                pair_indices=pair_indices, metric_callback=metric_callback, condition=condition,
            )
            records.append({"condition": condition, "recovery": recovery})
            completed.append(condition)
        # Phase 2 starts only after EVERY row of EVERY condition passes.
        curves = {name: [] for name in masks}
        for record in records:
            condition = record["condition"]
            mean = record["recovery"].mean()
            if not t.isfinite(mean).item():
                _stop_recovery("aggregation", pair_indices, "nonfinite_mean", condition)
            curves[condition[1]].append(mean.item())
    except RecoveryCohortFailure as failure:
        if failure.condition is None:
            failure.condition = condition
        failure.attempted_conditions = tuple(attempted)
        failure.completed_conditions = tuple(completed)
        failure.unattempted_conditions = tuple(c for c in condition_order if c not in attempted)
        raise
    return {
        "policy_version": "mini-vlm-checked-recovery-design-v1",
        "status": "complete_checked", "pair_indices": pair_indices,
        "clean_ld": clean_ld, "corrupt_ld": corrupt_ld, "gap": gap,
        "condition_order": condition_order, "rows": records, "curves": curves,
    }

```

</details>

## 5. Try It Yourself — break a control before touching a model

Predict the failing pair, stage and number of callback calls. Then change the literal fixture in your immediate tests:

- Reverse row 7's clean-versus-corrupt gap while row 9 has a zero gap. Both original indices should be reported together.
- Make row 9's callback result NaN in the last condition. No earlier means may run.
- Keep a tiny positive float64 gap and return a finite recovery greater than one. Both rows must remain.
- Use a reusable callback buffer. Later changes must not rewrite earlier saved rows.

For a valid control, display a two-row table with original pair ID, clean LD, corrupt LD, gap and recovery. Expected baseline gaps are 4 and 2. This engineering control table is not a Mini VLM scientific result.

**Reflect.** Why does removing a failed pair change the question? Why might a tiny positive gap produce an unstable but finite recovery? Does passing the checks prove object-token patching beats random background? Write your answer before reading the interpretation.

<details><summary>Interpretation</summary>

A valid denominator and finite output establish that this normalization is defined under the selected failure policy. They do not establish a faithful intervention, a correct callback formula, a trained model, or a causal advantage over controls. Large ratios with very small positive gaps deserve interpretation, not silent exclusion. Keep exposure status and fixed cohort identity in any later report.
</details>

## 6. Future learner run — explicit prerequisites, own callbacks

This section is deliberately separate from the tiny DEV controls. Use it only in an authorized, already prepared 12.3 session with your own completed functions and actual existing pair/model state. The original saved model/projector weights were not preserved as recoverable checkpoints; this notebook does not recreate them. If prerequisites are missing, stop here.

Bind `my_recovery` to your own completed `logit_diff_recovery`, then retain the existing callback tests. `pair_indices` must be assigned to the original selected pairs before measurements; for the unchanged complete `pairs` list its zero-based row indices are appropriate. Never derive IDs after filtering. Do not optimize the cohort from observed failures. Callbacks must not mutate experiment inputs.

The visible adapter below uses the existing 12.3 names: `patch_sets`, `ANSWER_POS`, `N_LAYERS`, `vlm`, `corrupt_embeds`, `clean_resids`, logits and answer IDs. It does not call, redefine or patch the original `patching_sweep`. Missing prerequisites raise normally rather than triggering setup, training or reference imports.

In [ ]:
# FUTURE LEARNER RUN ONLY — not part of the tiny CPU DEV controls.
# Clear any older success before a new attempt, so a failure cannot reuse its curve.
prospective_checked_recovery = None
my_recovery = logit_diff_recovery  # YOUR completed visible function, never an imported solution.
tests.test_logit_diff_recovery(my_recovery)  # Keep the original correctness tests.
my_pair_indices = tuple(range(len(pairs)))  # Complete original cohort, before any filtering.


def my_patch_logits(layer, mask_name):
    return patched_answer_logits(
        vlm, corrupt_embeds, clean_resids[layer], layer, patch_sets[mask_name], ANSWER_POS
    )


try:
    with t.no_grad():
        prospective_checked_recovery = checked_learner_sweep(
            layers=tuple(range(N_LAYERS)), masks=patch_sets, pair_indices=my_pair_indices,
            clean_logits=clean_logits, corrupt_logits=corrupt_logits,
            clean_ids=clean_ids, corrupt_ids=corrupt_ids,
            patch_logits_callback=my_patch_logits, metric_callback=my_recovery,
        )
except RecoveryCohortFailure as error:
    print("INCOMPLETE COHORT — no means or figure published")
    print("stage:", error.stage, "condition:", error.condition)
    print("invalid pairs:", error.invalid_pairs)
    print("completed:", error.completed_conditions, "unattempted:", error.unattempted_conditions)
    raise

In [ ]:
# FUTURE SUCCESS-ONLY DISPLAY. No saving or overwriting historical figures/reports.
if prospective_checked_recovery is None:
    raise RuntimeError("No complete successful checked sweep; do not plot a partial or older result")
checked = prospective_checked_recovery
if checked["status"] != "complete_checked":
    raise RuntimeError("The selected cohort has not passed completely")
print("pair_index | clean_LD | corrupt_LD | gap")
for row, pair_index in enumerate(checked["pair_indices"]):
    print(pair_index, checked["clean_ld"][row].item(), checked["corrupt_ld"][row].item(),
          checked["gap"][row].item())
# Use your already prepared matplotlib session. Every control curve remains visible.
prospective_figure, prospective_axis = plt.subplots()
for mask_name, values in checked["curves"].items():
    prospective_axis.plot(tuple(range(N_LAYERS)), values, label=mask_name)
prospective_axis.set(xlabel="Layer", ylabel="Recovery",
                     title="Prospective learner-run checked recovery — complete selected cohort")
prospective_axis.legend()
prospective_figure.tight_layout()

## Limitations and recording a later run

This addon changes prospective failure handling; it does not edit or revalidate the saved historical Mini reference. Keep the existing object, same-size random background, other-object, all-image, question and answer-position controls. Inspect the full gap table before interpreting the new curve. No result is promised.

Before any future scientific run, record the completed callback/addon source, original cohort IDs/order, conditions/order, model/checkpoint provenance, runtime/dtype and policy version. Any learner-saved receipt belongs to a new output location, never a historical report path. On failure mark the whole cohort incomplete, preserve the exception cause and attempted/unattempted conditions, and publish no partial means/figures. Serialize nonfinite diagnostics as explicit tags, never JSON NaN/Infinity numbers.

Tiny CPU DEV checks are engineering evidence only. Native reproduction, scientific validity and a human learner walkthrough remain separate. This source notebook ships with no execution counts or outputs; it is not an executed reference.